In [1]:
import pandas as pd

In [2]:
raw_curated_sheet=pd.read_excel('./Data/sge_data_for_qc/spliceai_benchmarking/20260506_CuratedSplicingTruthset.xlsx', sheet_name='curated_variants')
first_pass_vep=pd.read_csv('./Data/sge_data_for_qc/spliceai_benchmarking/20260507_CuratedSplicing_VEPAnnotations_firstpass.txt', sep='\t')

In [3]:
#initial merge

NM_TO_GENE = {
    "NM_002878": "RAD51D", "NM_058216": "RAD51C",
    "NM_007294": "BRCA1",  "NM_000059": "BRCA2",
    "NM_000051": "ATM",    "NM_007194": "CHEK2",
    "NM_024675": "PALB2",  "NM_004360": "CDH1",
    "NM_000249": "MLH1",   "NM_000251": "MSH2",
    "NM_000179": "MSH6",   "NM_000535": "PMS2",
    "NM_005732": "RAD50",  "NM_032043": "BRIP1",
    "NM_000038": "APC",    "NM_000546": "TP53",
    "NM_000143": "FH",     "NM_006767": "LZTR1",
    "NM_000297": "PKD2",   "NM_000314": "PTEN",
    "NM_000465": "BARD1",  "NM_182625": "GEN1",
}

GENE_TO_NM = {
    "RAD51D": "NM_002878.4", "RAD51C": "NM_058216.3",
    "BRCA1":  "NM_007294.4", "BRCA2":  "NM_000059.4",
    "ATM":    "NM_000051.4", "CHEK2":  "NM_007194.4",
    "PALB2":  "NM_024675.4", "CDH1":   "NM_004360.5",
    "MLH1":   "NM_000249.4", "MSH2":   "NM_000251.3",
    "MSH6":   "NM_000179.3", "PMS2":   "NM_000535.7",
    "RAD50":  "NM_005732.4", "BRIP1":  "NM_032043.3",
    "APC":    "NM_000038.6", "TP53":   "NM_000546.6",
    "FH":     "NM_000143.4", "LZTR1":  "NM_006767.4",
    "PKD2":   "NM_000297.4", "PTEN":   "NM_000314.8",
    "BARD1":  "NM_000465.4", "GEN1":   "NM_182625.3",
}

DROP_VARIANTS = {
    ("BRCA2", "c.7397C>T"),
    ("BRCA1", "c.190G>T"),
    ("TP53",  "c.786-60G>A"),
    ("ATM",   "c.5007-3T>A"),
    ("ATM",   "c.8787-13G>T"),
}

SPLICEAI_COLS = [
    "SpliceAI_pred_DP_AG", "SpliceAI_pred_DP_AL",
    "SpliceAI_pred_DP_DG", "SpliceAI_pred_DP_DL",
    "SpliceAI_pred_DS_AG", "SpliceAI_pred_DS_AL",
    "SpliceAI_pred_DS_DG", "SpliceAI_pred_DS_DL",
    "SpliceAI_pred_SYMBOL",
]

# --- assumes raw_curated_sheet (main sheet) and vep (VEP output) are already loaded ---
# vep should be loaded skipping ## comment lines e.g.:
# with open("vep_output.txt") as f:
#     lines = [l for l in f if not l.startswith("##")]
# vep = pd.read_csv(io.StringIO("".join(lines)), sep="\t", dtype=str)
# vep.columns = [c.lstrip("#") for c in vep.columns]

# clean main raw_curated_sheet
raw_curated_sheet["Gene"]   = raw_curated_sheet["Gene"].str.strip()
raw_curated_sheet["hgvs_c"] = raw_curated_sheet["hgvs_c"].str.strip()
raw_curated_sheet = raw_curated_sheet[raw_curated_sheet["Gene"].notna() & raw_curated_sheet["hgvs_c"].notna()].copy()

# drop problem variants
raw_curated_sheet = raw_curated_sheet[~raw_curated_sheet.apply(lambda r: (r["Gene"], r["hgvs_c"]) in DROP_VARIANTS, axis=1)].copy()

# build merge key
raw_curated_sheet["hgvs_full"] = raw_curated_sheet.apply(
    lambda r: f"{GENE_TO_NM[r['Gene']]}:{r['hgvs_c'].replace(' ', '')}"
    if r["Gene"] in GENE_TO_NM else None, axis=1
)

# filter VEP to correct gene only (removes off-target neighbors)
first_pass_vep["expected_gene"] = first_pass_vep["Uploaded_variation"].apply(
    lambda x: NM_TO_GENE.get(str(x).split(":")[0].rsplit(".", 1)[0])
)
vep_clean =first_pass_vep[first_pass_vep["SYMBOL"] == first_pass_vep["expected_gene"]].copy()

# deduplicate and select columns
keep = ["Uploaded_variation", "Location", "HGVSp", "STRAND"] + SPLICEAI_COLS
vep_slim = (vep_clean[keep]
            .drop_duplicates(subset="Uploaded_variation")
            .rename(columns={"Uploaded_variation": "hgvs_full",
                             "Location": "pos_vep",
                             "HGVSp": "hgvsp_vep"}))

# merge
merged = raw_curated_sheet.merge(vep_slim, on="hgvs_full", how="left")

# fill pos
merged["pos"] = merged["pos_vep"]

# fill aa: prefer existing, fall back to VEP HGVSp (strip transcript prefix)
existing_aa = merged["amino_acid_substitution"].fillna("").str.strip()
merged["amino_acid_substitution"] = existing_aa.where(
    existing_aa != "",
    merged["hgvsp_vep"].apply(
        lambda x: x.split(":")[-1] if pd.notna(x) and ":" in str(x) else (None if pd.isna(x) else x)
    )
)


merged['ref']=merged['hgvs_c'].transform(lambda x: x[-3])
merged['alt']=merged['hgvs_c'].transform(lambda x: x[-1])
merged['chrom']=merged['pos'].transform(lambda x: int(x.split(':')[0]))
merged['pos']=merged['pos'].transform(lambda x: int(x.split('-')[1]))

first_merge = merged
first_merge.head()

,Gene,pos,cds_pos,hgvs_c,hgvs_full,amino_acid_substitution,splice_consequence,source,pos_vep,hgvsp_vep,...,SpliceAI_pred_DP_DG,SpliceAI_pred_DP_DL,SpliceAI_pred_DS_AG,SpliceAI_pred_DS_AL,SpliceAI_pred_DS_DG,SpliceAI_pred_DS_DL,SpliceAI_pred_SYMBOL,ref,alt,chrom
0,RAD51D,35118621,NaN,c.145-2A>G,NM_002878.4:c.145-2A>G,-,abnormal,bueno_martinez_2021,17:35118621-35118621,-,...,-,-,-,-,-,-,-,A,G,17
1,RAD51D,35118495,NaN,c.263+6T>C,NM_002878.4:c.263+6T>C,-,normal,bueno_martinez_2021,17:35118495-35118495,-,...,-,-,-,-,-,-,-,T,C,17
2,RAD51D,35107368,NaN,c.343C>T,NM_002878.4:c.343C>T,-,abnormal,bueno_martinez_2021,17:35107368-35107368,-,...,-,-,-,-,-,-,-,C,T,17
3,RAD51D,35107364,NaN,c.345+2T>C,NM_002878.4:c.345+2T>C,-,abnormal,bueno_martinez_2021,17:35107364-35107364,-,...,-,-,-,-,-,-,-,T,C,17
4,RAD51D,35106987,NaN,c.480+1G>A,NM_002878.4:c.480+1G>A,-,abnormal,bueno_martinez_2021,17:35106987-35106987,-,...,-,-,-,-,-,-,-,G,A,17


In [4]:
rev_comp_dict = {'A': 'T',
                     'C':'G',
                     'T':'A',
                     'G':'C'
                     }

antisense_merged = first_merge[first_merge['STRAND']==-1].copy()

antisense_merged['ref']=antisense_merged['ref'].map(rev_comp_dict)
antisense_merged['alt']=antisense_merged['alt'].map(rev_comp_dict)

sense_merged = first_merge[first_merge['STRAND']==1].copy()

to_new_vcf = pd.concat([antisense_merged, sense_merged])
to_new_vcf = to_new_vcf.dropna(subset='alt')
to_new_vcf = to_new_vcf.sort_values(['chrom','pos'])
to_new_vcf.head()

,Gene,pos,cds_pos,hgvs_c,hgvs_full,amino_acid_substitution,splice_consequence,source,pos_vep,hgvsp_vep,...,SpliceAI_pred_DP_DG,SpliceAI_pred_DP_DL,SpliceAI_pred_DS_AG,SpliceAI_pred_DS_AL,SpliceAI_pred_DS_DG,SpliceAI_pred_DS_DL,SpliceAI_pred_SYMBOL,ref,alt,chrom
483,FH,241500601,NaN,c.1237-11C>G,NM_000143.4:c.1237-11C>G,-,intermediate,dragos_2022,1:241500601-241500601,-,...,-,-,-,-,-,-,-,G,C,1
521,MSH2,47407923,NaN,c.212-478T>G,NM_000251.3:c.212-478T>G,-,abnormal,van_der_klift_2015,2:47407923-47407923,-,...,-1,-2,0.00,0.00,1.00,0.00,MSH2,T,G,2
534,MSH2,47407923,NaN,c.212-478T>G,NM_000251.3:c.212-478T>G,abnormal,van_der_klift_2015,NaN,2:47407923-47407923,-,...,-1,-2,0.00,0.00,1.00,0.00,MSH2,T,G,2
469,MSH2,47408558,NaN,c.366+3A>G,NM_000251.3:c.366+3A>G,-,normal,karam_2019,2:47408558-47408558,-,...,-19,-3,0.00,0.00,0.00,0.00,MSH2,A,G,2
470,MSH2,47410375,NaN,c.645+3A>G,NM_000251.3:c.645+3A>G,-,normal,karam_2019,2:47410375-47410375,-,...,5,-3,0.00,0.00,0.00,0.00,MSH2,A,G,2


In [5]:

with open('./Data/sge_data_for_qc/spliceai_benchmarking/20260511_CuratedSplicing_CleanedVCF.vcf', 'w') as f:
    f.write("##fileformat=VCFv4.3\n")
    f.write("##reference=GRCh38\n")
    f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
    for _, row in to_new_vcf.iterrows():
        f.write(
            f"{row['chrom']}\t{row['pos']}\t.\t"
            f"{row['ref'].upper()}\t{row['alt'].upper()}\t.\tPASS\t.\n"
        )